In [22]:
# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

In [23]:
# ============================================================
# 2. LOAD DATA
# ============================================================

team_master = pd.read_csv(
    "team_master.csv"
)

player_master = pd.read_csv(
    "player_master.csv"
)

print(team_master.shape)
print(player_master.shape)


(96, 63)
(1937, 73)


In [24]:
# ============================================================
# 3. HELPERS
# ============================================================

def clean_numeric_df(df, feature_cols):
    df = df.copy()

    for col in feature_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

    return df


def get_position_group(position):
    position = str(position)

    if position == "Goalkeeper":
        return "Goalkeeper"

    if any(x in position for x in ["Back", "Center Back", "Wing Back"]):
        return "Defender"

    if any(x in position for x in ["Midfield", "Midfielder"]):
        return "Midfielder"

    if any(x in position for x in ["Wing", "Forward", "Striker"]):
        return "Forward"

    return "Other"

In [25]:
# ============================================================
# 4. TEAM SIMILARITY BASE
# ============================================================

team_features = [
    "shots_per_match",
    "passes_per_match",
    "carries_per_match",
    "xg_per_match",
    "goals_per_match",
    "avg_pass_completion_pct",
    "tempo",
    "directness",
    "pressing_intensity",
    "defensive_activity",
    "attack_score",
    "build_up_score",
    "defense_score",
    "directness_score"
]

team_similarity_base = team_master[
    [
        "competition_name",
        "season_name",
        "competition_short",
        "tournament_id",
        "team"
    ] + team_features
].copy()

team_similarity_base = clean_numeric_df(team_similarity_base, team_features)

print(team_similarity_base.shape)
team_similarity_base.head()

(96, 19)


,competition_name,season_name,competition_short,tournament_id,team,shots_per_match,passes_per_match,carries_per_match,xg_per_match,goals_per_match,avg_pass_completion_pct,tempo,directness,pressing_intensity,defensive_activity,attack_score,build_up_score,defense_score,directness_score
0,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,15.67,555.0,449.00,1.29,1.00,84.27,1004.00,2.82,144.67,58.67,64.48,97.50,15.00,62.50
1,African Cup of Nations,2023,AFCON,AFCON_2023,Angola,7.00,296.0,218.00,1.20,1.00,72.30,514.00,2.36,228.00,87.00,38.85,19.58,65.42,25.00
2,African Cup of Nations,2023,AFCON,AFCON_2023,Burkina Faso,7.00,266.0,196.00,1.26,2.00,71.80,462.00,2.63,167.00,60.00,53.02,13.75,35.42,54.17
3,African Cup of Nations,2023,AFCON,AFCON_2023,Cameroon,5.00,424.0,332.00,0.25,1.00,78.07,756.00,1.18,203.00,69.00,18.23,51.67,52.19,8.33
4,African Cup of Nations,2023,AFCON,AFCON_2023,Cape Verde Islands,17.00,461.0,361.33,2.85,1.33,80.28,822.33,3.69,203.00,89.33,87.60,64.17,84.90,79.17


In [26]:
# ============================================================
# 5. TEAM SIMILARITY MODEL
# ============================================================

scaler = StandardScaler()

team_scaled = scaler.fit_transform(
    team_similarity_base[team_features]
)

team_similarity_matrix = cosine_similarity(team_scaled)

team_similarity_rows = []

for i in range(len(team_similarity_base)):

    row_i = team_similarity_base.iloc[i]

    for j in range(len(team_similarity_base)):

        if i == j:
            continue

        row_j = team_similarity_base.iloc[j]

        similarity_score = team_similarity_matrix[i][j] * 100

        team_similarity_rows.append({
            "competition_name": row_i["competition_name"],
            "season_name": row_i["season_name"],
            "competition_short": row_i["competition_short"],
            "tournament_id": row_i["tournament_id"],
            "team": row_i["team"],

            "similar_competition_name": row_j["competition_name"],
            "similar_season_name": row_j["season_name"],
            "similar_competition_short": row_j["competition_short"],
            "similar_tournament_id": row_j["tournament_id"],
            "similar_team": row_j["team"],

            "similarity_score": similarity_score
        })

team_similarity = pd.DataFrame(team_similarity_rows)

team_similarity["similarity_rank"] = (
    team_similarity
    .groupby(["tournament_id", "team"])["similarity_score"]
    .rank(method="first", ascending=False)
)

team_similarity = team_similarity[
    team_similarity["similarity_rank"] <= 10
].copy()

team_similarity.head()

,competition_name,season_name,competition_short,tournament_id,team,similar_competition_name,similar_season_name,similar_competition_short,similar_tournament_id,similar_team,similarity_score,similarity_rank
5,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,African Cup of Nations,2023,AFCON,AFCON_2023,Côte d'Ivoire,77.044044,9.0
14,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,African Cup of Nations,2023,AFCON,AFCON_2023,Morocco,89.422969,1.0
23,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Copa America,2024,Copa América,Copa América_2024,Argentina,77.339797,8.0
33,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Copa America,2024,Copa América,Copa América_2024,Panama,79.072777,6.0
41,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,FIFA World Cup,2022,World Cup,World Cup_2022,Belgium,74.140202,10.0


In [27]:
# ============================================================
# 6. PLAYER SIMILARITY BASE
# ============================================================

player_master["position_group"] = player_master["position"].apply(get_position_group)

player_features = [
    "xg_per_match",
    "shots_per_match",
    "passes_per_match",
    "completed_passes_per_match",
    "pass_completion_pct",
    "carries_per_match",
    "pressures_per_match",
    "recoveries_per_match",
    "duels_per_match",
    "interceptions_per_match",
    "blocks_per_match",
    "clearances_per_match",
    "attacking_volume",
    "passing_volume",
    "carrying_volume",
    "pressing_volume",
    "ball_winning_volume",
    "defensive_volume",
    "importance_score"
]

player_features = [
    col for col in player_features
    if col in player_master.columns
]

player_similarity_base = player_master[
    [
        "competition_name",
        "season_name",
        "competition_short",
        "tournament_id",
        "team",
        "player",
        "position",
        "position_group",
        "general_role"
    ] + player_features
].copy()

player_similarity_base = clean_numeric_df(player_similarity_base, player_features)

print(player_similarity_base.shape)
player_similarity_base.head()

(1937, 28)


,competition_name,season_name,competition_short,tournament_id,team,player,position,position_group,general_role,xg_per_match,...,interceptions_per_match,blocks_per_match,clearances_per_match,attacking_volume,passing_volume,carrying_volume,pressing_volume,ball_winning_volume,defensive_volume,importance_score
0,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Adam Ounas,Right Wing,Forward,Balanced Role,0.01,...,0.0,0.00,0.00,0.27,13.33,19.00,3.00,2.33,4.33,37.68
1,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Anthony Louis Mandréa,Goalkeeper,Goalkeeper,Goalkeeper,0.00,...,0.0,0.00,0.00,0.00,23.93,13.33,0.00,4.33,4.33,47.74
2,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Aïssa Mandi,Right Center Back,Defender,Tempo Controller,0.16,...,0.0,2.67,2.33,0.49,69.40,55.67,2.67,4.00,10.33,75.95
3,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Baghdad Bounedjah,Center Forward,Forward,Box Threat,0.63,...,0.0,1.33,1.00,1.58,9.07,9.00,14.00,4.00,18.33,53.12
4,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Fares Chaïbi,Left Wing,Forward,Box Threat,0.05,...,0.0,0.50,0.00,0.63,31.30,26.50,7.00,2.00,9.00,60.57


In [28]:
# ============================================================
# 7. PLAYER SIMILARITY MODEL BY POSITION GROUP
# ============================================================

top_k = 15
similarity_rows = []

for position_group, group_df in player_similarity_base.groupby("position_group"):

    group_df = group_df.reset_index(drop=True).copy()

    if len(group_df) < 2:
        continue

    scaler = StandardScaler()

    player_scaled = scaler.fit_transform(
        group_df[player_features]
    )

    similarity_matrix = cosine_similarity(player_scaled)

    similarity_df = pd.DataFrame(
        similarity_matrix,
        index=group_df.index,
        columns=group_df.index
    )

    for idx in similarity_df.index:

        row_i = group_df.loc[idx]

        similarities = (
            similarity_df
            .loc[idx]
            .drop(idx)
            .sort_values(ascending=False)
            .head(top_k)
        )

        for similar_idx, score in similarities.items():

            row_j = group_df.loc[similar_idx]

            similarity_rows.append({
                "competition_name": row_i["competition_name"],
                "season_name": row_i["season_name"],
                "competition_short": row_i["competition_short"],
                "tournament_id": row_i["tournament_id"],
                "team": row_i["team"],
                "player": row_i["player"],
                "position": row_i["position"],
                "position_group": row_i["position_group"],
                "general_role": row_i["general_role"],

                "similar_competition_name": row_j["competition_name"],
                "similar_season_name": row_j["season_name"],
                "similar_competition_short": row_j["competition_short"],
                "similar_tournament_id": row_j["tournament_id"],
                "similar_team": row_j["team"],
                "similar_player": row_j["player"],
                "similar_position": row_j["position"],
                "similar_position_group": row_j["position_group"],
                "similar_general_role": row_j["general_role"],

                "similarity_score": score * 100
            })

player_similarity = pd.DataFrame(similarity_rows)

player_similarity["similarity_rank"] = (
    player_similarity
    .groupby(["tournament_id", "team", "player"])["similarity_score"]
    .rank(method="first", ascending=False)
)

player_similarity.head()

,competition_name,season_name,competition_short,tournament_id,team,player,position,position_group,general_role,similar_competition_name,similar_season_name,similar_competition_short,similar_tournament_id,similar_team,similar_player,similar_position,similar_position_group,similar_general_role,similarity_score,similarity_rank
0,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Aïssa Mandi,Right Center Back,Defender,Tempo Controller,FIFA World Cup,2022,World Cup,World Cup_2022,England,John Stones,Right Center Back,Defender,Tempo Controller,90.652278,1.0
1,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Aïssa Mandi,Right Center Back,Defender,Tempo Controller,African Cup of Nations,2023,AFCON,AFCON_2023,Côte d'Ivoire,Obite Evan Ndicka,Left Center Back,Defender,Tempo Controller,88.019346,2.0
2,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Aïssa Mandi,Right Center Back,Defender,Tempo Controller,FIFA World Cup,2022,World Cup,World Cup_2022,Germany,Antonio Rüdiger,Left Center Back,Defender,Box Threat,86.470705,3.0
3,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Aïssa Mandi,Right Center Back,Defender,Tempo Controller,FIFA World Cup,2022,World Cup,World Cup_2022,Brazil,Marcos Aoás Corrêa,Right Center Back,Defender,Tempo Controller,85.970593,4.0
4,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Aïssa Mandi,Right Center Back,Defender,Tempo Controller,UEFA Euro,2024,Euro,Euro_2024,Spain,Aymeric Laporte,Left Center Back,Defender,Tempo Controller,85.090755,5.0


In [29]:
# ============================================================
# 8. ROUND AND SAVE
# ============================================================

team_similarity["similarity_score"] = (
    team_similarity["similarity_score"]
    .round(2)
)

player_similarity["similarity_score"] = (
    player_similarity["similarity_score"]
    .round(2)
)

team_similarity.to_csv(
    "team_similarity.csv",
    index=False
)

player_similarity.to_csv(
    "player_similarity.csv",
    index=False
)

print("team_similarity.csv saved successfully!")
print("player_similarity.csv saved successfully!")

print("Team similarity:", team_similarity.shape)
print("Player similarity:", player_similarity.shape)

team_similarity.csv saved successfully!
player_similarity.csv saved successfully!
Team similarity: (960, 12)
Player similarity: (28995, 20)
